# Preprocessing GIS Network Data for NetLogo

This notebook prepares the Amsterdam road network and third-place datasets for use in the NetLogo simulation model. It loads cleaned node, edge, and place CSV files, removes invalid road links such as self-loops, missing node references, unused nodes, and duplicated undirected edges. The notebook then isolates the largest connected road-network component so that agents move within one continuous network rather than disconnected fragments. Finally, the road-node coordinates are rescaled into the simplified NetLogo world extent, allowing the real Amsterdam street network to be represented inside the simulation environment.

# Preprocessing GIS Data

In [17]:
import pandas as pd
from pathlib import Path

base = Path("/Users/tonyvo/Desktop/Thesis/netlogo/data")

files = {
    "nodes": base / "roads" / "roads_nodes_clean.csv",
    "edges": base / "roads" / "edges_clean.csv",
    "places": base / "places" / "places_clean.csv",
}

nodes_df = pd.read_csv(files["nodes"])
edges_df = pd.read_csv(files["edges"])
places_df = pd.read_csv(files["places"])

print(nodes_df.head())
print(nodes_df.shape)

print(edges_df.head())
print(edges_df.shape)

print(places_df.head())
print(places_df.shape)

# Avoid working with original file
nodes = nodes_df.copy()
edges = edges_df.copy()
places = places_df.copy()


   node_id            x            y
0        1  125951.0740  475615.8302
1        3  113223.2327  476246.3143
2        4  113598.8932  476458.6950
3        5  125816.9304  475188.7586
4        6  125539.0609  476460.8473
(173095, 3)
    edge_id  from_id  to_id  Shape_Length
0  118693.0        1      1     42.434735
1  118694.0        3      4    431.576976
2  118695.0        5      6   1389.800573
3  118696.0        4      8     48.283123
4  118697.0        9     10    181.908961
(242649, 4)
        id            x            y
0  4366722  123203.2097  485916.9237
1  4376560  122407.2095  486772.2775
2  4754220  119464.0840  484254.2656
3  4842719  118474.6676  485395.8845
4  4842723  118506.8791  485147.9962
(3702, 3)


## Remove bad edges
Remove edges with self loops, NA values, references to nodes that don't exist.

In [18]:
# make sure ID columns are numeric
nodes["node_id"] = pd.to_numeric(nodes["node_id"], errors="coerce")
edges["edge_id"] = pd.to_numeric(edges["edge_id"], errors="coerce")
edges["from_id"] = pd.to_numeric(edges["from_id"], errors="coerce")
edges["to_id"] = pd.to_numeric(edges["to_id"], errors="coerce")
edges["length_m"] = pd.to_numeric(edges["Shape_Length"], errors="coerce")

# Drop NAs
edges = edges.dropna(subset=["edge_id", "from_id", "to_id", "Shape_Length"]).copy()

# Make ID integers
nodes["node_id"] = nodes["node_id"].astype(int)
edges["edge_id"] = edges["edge_id"].astype(int)
edges["from_id"] = edges["from_id"].astype(int)
edges["to_id"] = edges["to_id"].astype(int)

In [19]:
# 1) remove self-loops
self_loops = edges[edges["from_id"] == edges["to_id"]].copy()
edges_no_loops = edges[edges["from_id"] != edges["to_id"]].copy()

print("\nSelf-loops removed:", len(self_loops))

# 2) remove edges with bad node references
valid_nodes = set(nodes["node_id"])

bad_from = ~edges_no_loops["from_id"].isin(valid_nodes)
bad_to = ~edges_no_loops["to_id"].isin(valid_nodes)
bad_edges = edges_no_loops[bad_from | bad_to].copy()

edges_valid = edges_no_loops[~(bad_from | bad_to)].copy()

print("Edges with missing from/to node removed:", len(bad_edges))



Self-loops removed: 327
Edges with missing from/to node removed: 0


In [20]:
out_no_self_loops = base / "roads" / "edges_no_self_loops.csv"
edges_no_loops.to_csv(out_no_self_loops, index=False)

## Remove Bad Nodes and Duplicated Undirected Links

In [23]:
nodes = pd.read_csv(base / "roads" / "roads_nodes_clean.csv")
edges = pd.read_csv(base / "roads" / "edges_no_self_loops.csv")

used_node_ids = set(edges["from_id"]).union(set(edges["to_id"]))

nodes_trimmed = nodes[nodes["node_id"].isin(used_node_ids)].copy()

print("Original nodes:", len(nodes))
print("Trimmed nodes:", len(nodes_trimmed))
print("Removed unused nodes:", len(nodes) - len(nodes_trimmed))

nodes_trimmed.to_csv(base / "roads" / "roads_nodes_clean.csv", index=False )

Original nodes: 173095
Trimmed nodes: 173011
Removed unused nodes: 84


In [27]:
edges["pair"] = edges.apply(lambda r: tuple(sorted((r["from_id"], r["to_id"]))), axis=1)
dup_pairs = edges[edges["pair"].duplicated(keep=False)].sort_values("pair")

print("Rows involved in duplicate undirected pairs:", len(dup_pairs))

edges_nodup = edges.drop_duplicates(subset=["pair"]).drop(columns=["pair"]).copy()

print("Original edges:", len(edges))
print("After removing undirected duplicates:", len(edges_nodup))

edges_nodup.to_csv(base / "roads" / "edges_nodup.csv", index=False)

Rows involved in duplicate undirected pairs: 3510
Original edges: 242322
After removing undirected duplicates: 240557


# Only Keep Largest Component

In [30]:
import networkx as nx

nodes = pd.read_csv(base / "roads" / "roads_nodes_clean.csv")
edges = pd.read_csv(base / "roads" / "edges_nodup.csv")

G= nx.Graph()

for _, row in nodes.iterrows():
    G.add_node(int(row["node_id"]))

for _, row in edges.iterrows():
    G.add_edge(int(row["from_id"]), int(row["to_id"]), edge_id=int(row["edge_id"]), length_m=float(row["length_m"]))

components = list(nx.connected_components(G))
components = sorted(components, key=len, reverse=True)

largest_component = components[0]
print("Number of connected components:", len(components))
print("Largest component size:", len(largest_component))

nodes_main = nodes[nodes["node_id"].isin(largest_component)].copy()
edges_main = edges[
    edges["from_id"].isin(largest_component) & edges["to_id"].isin(largest_component)
].copy()

print("Nodes in main component:", len(nodes_main))
print("Edges in main component:", len(edges_main))

nodes_main.to_csv(base / "roads" / "nodes_main_component.csv", index=False)
edges_main.to_csv(base / "roads" / "edges_main_component.csv", index=False)


Number of connected components: 596
Largest component size: 171336
Nodes in main component: 171336
Edges in main component: 239418


# Normalise to NetLogo World


In [32]:
nodes = pd.read_csv(base / "roads" / "nodes_main_component.csv")

min_x = nodes["x"].min()
max_x = nodes["x"].max()
min_y = nodes["y"].min()
max_y = nodes["y"].max()

data_width = max_x - min_x
data_height = max_y - min_y

nl_min_x, nl_max_x = -100, 100
nl_min_y, nl_max_y = -50, 50

world_width = nl_max_x - nl_min_x
world_height = nl_max_y - nl_min_y

scale = min(world_width / data_width, world_height / data_height)

x_mid_data = (min_x + max_x) / 2
y_mid_data = (min_y + max_y) / 2

x_mid_world = (nl_min_x + nl_max_x) / 2
y_mid_world = (nl_min_y + nl_max_y) / 2

nodes_scaled = nodes[["node_id"]].copy()
nodes_scaled["x"] = (nodes["x"] - x_mid_data) * scale + x_mid_world
nodes_scaled["y"] = (nodes["y"] - y_mid_data) * scale + y_mid_world

nodes_scaled.to_csv(base / "nodes_scaled_for_netlogo.csv", index=False)